# SC2079 symbol card classification (YOLO26-cls)

Fine-tunes a pretrained YOLO26-nano **classifier** directly on the ImageFolder captures in `dataset/{train,test}/<id>_<slug>/`.
No `autolabel.py` step: the folder name is the label, and a classifier needs no bounding boxes.

Trade-off vs. `train.ipynb` (detector): you get the image ID but not the card's location in the frame.

Ultralytics uses `dataset/test/` as the validation split because there is no `dataset/val/`.

In [1]:
import sys
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
device

'cuda'

In [2]:
ROOT = Path.cwd()
DATASET = ROOT / "dataset"  # ImageFolder root: train/ and test/

for split in ("train", "test"):
    counts = {d.name: sum(1 for p in d.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
              for d in sorted((DATASET / split).iterdir()) if d.is_dir()}
    print(f"{split}: {sum(counts.values())} images, {len(counts)} classes")
    for name, n in counts.items():
        print(f"    {name:<10} {n}")

train: 282 images, 10 classes
    11_1       23
    12_2       26
    13_3       32
    14_4       33
    15_5       29
    16_6       26
    17_7       41
    18_8       26
    19_9       28
    40_stop    18
test: 70 images, 10 classes
    11_1       6
    12_2       7
    13_3       7
    14_4       8
    15_5       7
    16_6       7
    17_7       10
    18_8       7
    19_9       7
    40_stop    4


In [3]:
model = YOLO("yolo26n-cls.pt")  # classification task, pretrained on ImageNet

# Classification augmentation in Ultralytics is torchvision-style: random resized
# crop (scale), HSV jitter, random erasing and RandAugment. degrees/translate/shear
# are detection-only and ignored here. fliplr stays 0: mirroring turns digits and
# left/right arrow cards into the wrong class.
results = model.train(
    data=DATASET,
    epochs=80,
    imgsz=640,
    batch=32,
    scale=0.5,        # random-resized-crop keeps 50-100% of the image
    hsv_v=0.5,        # brightness jitter (lighting varies on the robot)
    erasing=0.2,
    fliplr=0.0,       # NO horizontal flip (see above)
    device=device,
    seed=42,
    project=ROOT / "runs",
    name="cls",
)
BEST = Path(results.save_dir) / "weights" / "best.pt"
BEST

New https://pypi.org/project/ultralytics/8.4.154 available 😃 Update with 'pip install -U ultralytics'


Ultralytics 8.4.153 🚀 Python-3.11.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 4000 Blackwell, 23986MiB)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/image-rec/dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.2, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=cls-2, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=Tr

train: /workspace/image-rec/dataset/train... found 282 images in 10 classes ✅ 


val: None...


...

               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 1.5it/s 1.3s

                   all          1          1


Speed: 0.2ms preprocess, 18.2ms inference, 0.0ms loss, 0.0ms postprocess per image


Results saved to /workspace/image-rec/runs/cls-2


PosixPath('/workspace/image-rec/runs/cls-2/weights/best.pt')

### Evaluate

Top-1 accuracy on `dataset/test/`, then a per-class breakdown and the list of misclassified images.

In [4]:
model = YOLO(BEST)
metrics = model.val(data=DATASET, device=device, project=ROOT / "runs", name="cls-val")
print(f"top1: {metrics.top1:.3f}   top5: {metrics.top5:.3f}")

Ultralytics 8.4.153 🚀 Python-3.11.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 4000 Blackwell, 23986MiB)


YOLO26n-cls summary (fused): 47 layers, 1,538,834 parameters, 0 gradients, 3.2 GFLOPs


WARNING ⚠️ Dataset 'split=val' not found, using 'split=test' instead.


train: /workspace/image-rec/dataset/train... found 282 images in 10 classes ✅ 


val: /workspace/image-rec/dataset/test... found 70 images in 10 classes ✅ 


...

               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 5/5 1.9it/s 2.7s

                   all          1          1


Speed: 8.5ms preprocess, 18.9ms inference, 0.0ms loss, 0.0ms postprocess per image


Results saved to /workspace/image-rec/runs/cls-val-2


top1: 1.000   top5: 1.000


In [5]:
test_images = sorted(p for p in (DATASET / "test").rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
preds = model.predict([str(p) for p in test_images], imgsz=640, device=device, verbose=False)

per_class = {}
wrong = []
for path, res in zip(test_images, preds):
    truth = path.parent.name
    guess = model.names[int(res.probs.top1)]
    conf = float(res.probs.top1conf)
    n, k = per_class.get(truth, (0, 0))
    per_class[truth] = (n + 1, k + (guess == truth))
    if guess != truth:
        wrong.append((path.name, truth, guess, conf))

total = sum(n for n, _ in per_class.values())
correct = sum(k for _, k in per_class.values())
print(f"overall: {correct}/{total} = {correct / total:.3f}")
for name, (n, k) in sorted(per_class.items()):
    print(f"    {name:<10} {k}/{n}")
print(f"\n{len(wrong)} misclassified:")
for name, truth, guess, conf in wrong:
    print(f"    {name}: {truth} -> {guess} ({conf:.2f})")

overall: 70/70 = 1.000
    11_1       6/6
    12_2       7/7
    13_3       7/7
    14_4       8/8
    15_5       7/7
    16_6       7/7
    17_7       10/10
    18_8       7/7
    19_9       7/7
    40_stop    4/4

0 misclassified:


In [6]:
# snapshot the deployment weights
# prediction class -> image ID for the task: image_id = int(model.names[cls].split("_")[0])
import shutil

deployed = ROOT / "weights" / "best_cls.pt"
deployed.parent.mkdir(exist_ok=True)
shutil.copy2(BEST, deployed)
print(f"copied {BEST} -> {deployed}")

copied /workspace/image-rec/runs/cls-2/weights/best.pt -> /workspace/image-rec/weights/best_cls.pt
